# Mechanistic Interpretability on GPT-2 (Small)

This notebook explores the internals of GPT-2 Small (124M) using **TransformerLens**.

### Techniques covered:
1. **Setup & Model Loading** — load GPT-2 with TransformerLens hooks
2. **Attention Pattern Visualization** — what do attention heads attend to?
3. **Logit Lens / Probing** — how predictions evolve across layers
4. **Activation Patching / Causal Tracing** — which components cause specific predictions?
5. **Sparse Autoencoder (SAE) Features** — decompose MLP activations into sparse, interpretable features

---
## 1. Setup & Model Loading

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import pandas as pd
from tqdm import tqdm
import einops
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

import transformer_lens
from transformer_lens import HookedTransformer, utils
from transformer_lens.hook_points import HookPoint

# Try to import circuitsvis for rich attention visualization
try:
    import circuitsvis as cv
    HAS_CV = True
except ImportError:
    HAS_CV = False
    print("circuitsvis not found — will use matplotlib fallback for attention viz")

print(f"torch version      : {torch.__version__}")
print(f"transformer_lens   : {transformer_lens.__version__}")
device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"device             : {device}")

In [ ]:
# Load GPT-2 Small via TransformerLens
# TransformerLens wraps the model with named hook points at every residual stream,
# attention output, MLP output, etc.
model = HookedTransformer.from_pretrained('gpt2', device=device)
model.eval()

print(f"\nModel: GPT-2 Small")
print(f"  Layers      : {model.cfg.n_layers}")
print(f"  Heads/layer : {model.cfg.n_heads}")
print(f"  d_model     : {model.cfg.d_model}")
print(f"  d_mlp       : {model.cfg.d_mlp}")
print(f"  d_head      : {model.cfg.d_head}")
print(f"  Vocab size  : {model.cfg.d_vocab}")
print(f"  Context len : {model.cfg.n_ctx}")

In [ ]:
# Explore available hook points (activation names)
# These are the named locations where we can intercept or patch activations
all_hooks = model.hook_dict.keys()
print("Sample hook points (first 30):")
for name in list(all_hooks)[:30]:
    print(f"  {name}")
print(f"\nTotal hook points: {len(list(all_hooks))}")

In [ ]:
# Helper: tokenize a prompt and show tokens
def tokenize_and_show(prompt: str):
    tokens = model.to_tokens(prompt)  # shape: [1, seq_len]
    token_strs = model.to_str_tokens(prompt)
    print(f"Prompt  : {repr(prompt)}")
    print(f"Tokens  : {tokens[0].tolist()}")
    print(f"Decoded : {token_strs}")
    return tokens, token_strs

# Run a forward pass with cache — this stores all intermediate activations
PROMPT = "The Eiffel Tower is located in the city of"
tokens, token_strs = tokenize_and_show(PROMPT)

logits, cache = model.run_with_cache(tokens)

# Top-5 predictions for the next token
next_token_logits = logits[0, -1]  # last position
top5 = torch.topk(next_token_logits, 5)
print("\nTop-5 next token predictions:")
for tok_id, logit in zip(top5.indices, top5.values):
    print(f"  {repr(model.to_single_str_token(tok_id.item())):15s}  logit={logit.item():.2f}")

---
## 2. Attention Pattern Visualization

Attention patterns show which tokens each head attends to when producing its output.
Key patterns to look for:
- **Previous token heads** — attend mainly to position-1
- **Induction heads** — copy a token that followed a repeated prefix
- **Diagonal heads** — attend to self (current position)
- **Global heads** — attend broadly across the sequence

In [ ]:
def plot_attention_patterns_matplotlib(cache, tokens_str, layer: int, heads=None, figsize=(20, 12)):
    """
    Plot attention patterns for all (or selected) heads in a given layer.
    cache : ActivationCache from model.run_with_cache
    tokens_str : list of string tokens
    layer : which layer to visualize
    heads : list of head indices to plot (None = all 12)
    """
    # attn_pattern shape: [batch, n_heads, seq_len, seq_len]
    attn = cache['pattern', layer][0].cpu().numpy()  # [n_heads, seq, seq]
    n_heads = attn.shape[0]
    heads_to_plot = heads if heads is not None else list(range(n_heads))
    ncols = 4
    nrows = (len(heads_to_plot) + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes = axes.flatten()

    for idx, h in enumerate(heads_to_plot):
        ax = axes[idx]
        im = ax.imshow(attn[h], cmap='viridis', vmin=0, vmax=attn[h].max())
        ax.set_title(f'L{layer}H{h}', fontsize=10)
        ax.set_xticks(range(len(tokens_str)))
        ax.set_xticklabels(tokens_str, rotation=60, ha='right', fontsize=7)
        ax.set_yticks(range(len(tokens_str)))
        ax.set_yticklabels(tokens_str, fontsize=7)
        plt.colorbar(im, ax=ax, fraction=0.046)

    for ax in axes[len(heads_to_plot):]:
        ax.axis('off')

    fig.suptitle(f'Attention Patterns — Layer {layer}', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()


# Visualize layer 0 and layer 5 attention patterns
for layer in [0, 5, 10]:
    plot_attention_patterns_matplotlib(cache, token_strs, layer=layer)

In [ ]:
# --- Induction head detection ---
# Induction heads attend to the token that followed the last occurrence of
# the current token (i.e. pattern [A][B]...[A] -> [B]).
# Classic test: feed a repeated random sequence and look for the off-diagonal stripe.

def detect_induction_heads(model, seq_len=50, seed=42):
    """
    Generate a repeated random token sequence [rand_tokens, rand_tokens]
    and score each head by how strongly it attends to the induction position.
    Returns an (n_layers, n_heads) score matrix.
    """
    torch.manual_seed(seed)
    rand_tokens = torch.randint(1000, 10000, (1, seq_len), device=device)
    rep_tokens = torch.cat([rand_tokens, rand_tokens], dim=1)   # [1, 2*seq_len]

    _, rep_cache = model.run_with_cache(rep_tokens)

    induction_scores = torch.zeros(model.cfg.n_layers, model.cfg.n_heads)
    for layer in range(model.cfg.n_layers):
        attn = rep_cache['pattern', layer][0]  # [n_heads, 2*seq_len, 2*seq_len]
        # The induction stripe is at position [seq+i, i+1] for i in [0, seq_len-1]
        for head in range(model.cfg.n_heads):
            stripe = attn[head, seq_len:, 1:seq_len].diag().mean().item()
            induction_scores[layer, head] = stripe

    return induction_scores

induction_scores = detect_induction_heads(model)

plt.figure(figsize=(10, 5))
sns.heatmap(induction_scores.numpy(), annot=True, fmt='.2f', cmap='Reds',
            xticklabels=[f'H{h}' for h in range(model.cfg.n_heads)],
            yticklabels=[f'L{l}' for l in range(model.cfg.n_layers)])
plt.title('Induction Head Scores (higher = stronger induction behaviour)')
plt.xlabel('Head')
plt.ylabel('Layer')
plt.tight_layout()
plt.show()

# Top induction heads
flat = induction_scores.flatten()
top_k = torch.topk(flat, 5)
print("Top-5 induction heads:")
for score, idx in zip(top_k.values, top_k.indices):
    l, h = divmod(idx.item(), model.cfg.n_heads)
    print(f"  L{l}H{h}  score={score.item():.3f}")

---
## 3. Logit Lens & Linear Probing

**Logit Lens**: project each layer's residual stream through the final unembedding matrix to see how the model's prediction evolves layer by layer.

**Linear Probing**: train a lightweight linear classifier on a layer's residual stream to test whether a specific concept is linearly encoded.

In [ ]:
def logit_lens(model, cache, token_strs, top_k=5):
    """
    For each layer's residual stream output, apply layer norm + unembed
    and return the top-k predicted tokens at each (layer, position).
    """
    n_layers = model.cfg.n_layers
    seq_len = len(token_strs)
    results = {}

    # We use the 'resid_post' hook (residual stream after each layer)
    for layer in range(n_layers):
        resid = cache['resid_post', layer][0]        # [seq_len, d_model]
        # Apply final layer norm and unembed
        normed = model.ln_final(resid)               # [seq_len, d_model]
        logits = model.unembed(normed)               # [seq_len, d_vocab]
        probs = torch.softmax(logits, dim=-1)

        top = torch.topk(probs, top_k, dim=-1)
        results[layer] = {
            'top_tokens': [[model.to_single_str_token(t.item()) for t in row]
                           for row in top.indices],
            'top_probs': top.values.detach().cpu().numpy(),
        }
    return results

ll_results = logit_lens(model, cache, token_strs)

# --- Visualize: probability of the final answer token across layers ---
# What we expect: probability of "Paris" should rise in later layers

final_answer = " Paris"
final_answer_id = model.to_single_token(final_answer)

probs_per_layer = []
for layer in range(model.cfg.n_layers):
    resid = cache['resid_post', layer][0, -1]      # last position
    normed = model.ln_final(resid.unsqueeze(0))
    logits = model.unembed(normed)[0]
    prob = torch.softmax(logits, dim=-1)[final_answer_id].item()
    probs_per_layer.append(prob)

plt.figure(figsize=(10, 4))
plt.plot(range(model.cfg.n_layers), probs_per_layer, marker='o', linewidth=2, color='steelblue')
plt.xlabel('Layer')
plt.ylabel(f'P("{final_answer}")')
plt.title(f'Logit Lens: Probability of "{final_answer}" at final token position, across layers')
plt.xticks(range(model.cfg.n_layers))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# --- Heatmap: top-1 predicted token at each (layer, position) ---
top1_matrix = [[ll_results[l]['top_tokens'][p][0] for p in range(len(token_strs))]
               for l in range(model.cfg.n_layers)]
top1_probs  = [[ll_results[l]['top_probs'][p][0]  for p in range(len(token_strs))]
               for l in range(model.cfg.n_layers)]

fig, ax = plt.subplots(figsize=(14, 6))
im = ax.imshow(top1_probs, aspect='auto', cmap='YlOrRd')
ax.set_xticks(range(len(token_strs)))
ax.set_xticklabels(token_strs, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(model.cfg.n_layers))
ax.set_yticklabels([f'L{l}' for l in range(model.cfg.n_layers)], fontsize=9)
ax.set_title('Logit Lens — Top-1 predicted token probability (colour) and label (text)')
plt.colorbar(im, ax=ax, label='Probability')

for l in range(model.cfg.n_layers):
    for p in range(len(token_strs)):
        ax.text(p, l, top1_matrix[l][p], ha='center', va='center', fontsize=6,
                color='black' if top1_probs[l][p] < 0.5 else 'white')

plt.tight_layout()
plt.show()

In [ ]:
# --- Linear Probing: does a layer's residual stream encode "city" vs "country"? ---
# We construct a small dataset of city and country name prompts,
# extract mid-layer residuals at the final token, and train a linear classifier.

CITY_PROMPTS    = [
    "The Eiffel Tower is in the city of Paris",
    "Big Ben is in the city of London",
    "The Colosseum is in the city of Rome",
    "The Sagrada Familia is in the city of Barcelona",
    "Times Square is in the city of New York",
    "The Opera House is in the city of Sydney",
    "The Kremlin is in the city of Moscow",
    "The Alhambra is in the city of Granada",
]
COUNTRY_PROMPTS = [
    "The Eiffel Tower is in the country of France",
    "Big Ben is in the country of England",
    "The Colosseum is in the country of Italy",
    "The Sagrada Familia is in the country of Spain",
    "Times Square is in the country of America",
    "The Opera House is in the country of Australia",
    "The Kremlin is in the country of Russia",
    "The Alhambra is in the country of Spain",
]

labels = [0]*len(CITY_PROMPTS) + [1]*len(COUNTRY_PROMPTS)
all_prompts = CITY_PROMPTS + COUNTRY_PROMPTS

def get_final_token_resid(model, prompts, layer):
    """Return residual stream at the final token position for each prompt."""
    vecs = []
    for p in prompts:
        toks = model.to_tokens(p)
        _, c = model.run_with_cache(toks, names_filter=f'blocks.{layer}.hook_resid_post')
        vec = c[f'blocks.{layer}.hook_resid_post'][0, -1].detach().cpu().numpy()
        vecs.append(vec)
    return np.stack(vecs)

probe_accs = []
for layer in range(model.cfg.n_layers):
    X = get_final_token_resid(model, all_prompts, layer)
    clf = LogisticRegression(max_iter=500, C=1.0)
    clf.fit(X, labels)
    acc = accuracy_score(labels, clf.predict(X))
    probe_accs.append(acc)

plt.figure(figsize=(10, 4))
plt.plot(range(model.cfg.n_layers), probe_accs, marker='s', linewidth=2, color='coral')
plt.axhline(0.5, linestyle='--', color='gray', label='chance')
plt.xlabel('Layer')
plt.ylabel('Probe accuracy')
plt.title('Linear Probe: city vs. country at final token — across layers')
plt.xticks(range(model.cfg.n_layers))
plt.ylim(0.4, 1.05)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Best probe layer: L{np.argmax(probe_accs)}  acc={max(probe_accs):.2f}")

---
## 4. Activation Patching / Causal Tracing

**Activation patching** asks: *if I replace the activation at (layer L, position P) with the one from a "corrupted" run, how much does the model's answer change?*

This is the key technique from [Meng et al. 2022 (ROME)](https://rome.baulab.info/) and lets us localize where a fact is stored.

**Setup:**
- **Clean run**: `"The Eiffel Tower is located in the city of"` → expects ` Paris`
- **Corrupted run**: same prompt but with `"Eiffel Tower"` replaced by random token noise
- We patch one activation at a time from the clean cache into the corrupted run and measure how much probability of ` Paris` is restored.

In [ ]:
CLEAN_PROMPT  = "The Eiffel Tower is located in the city of"
TARGET_TOKEN  = " Paris"
target_id     = model.to_single_token(TARGET_TOKEN)

clean_tokens  = model.to_tokens(CLEAN_PROMPT)
clean_token_strs = model.to_str_tokens(CLEAN_PROMPT)
seq_len = clean_tokens.shape[1]

# --- Clean run ---
clean_logits, clean_cache = model.run_with_cache(clean_tokens)
clean_prob = torch.softmax(clean_logits[0, -1], dim=-1)[target_id].item()
print(f"Clean P('{TARGET_TOKEN}') = {clean_prob:.4f}")

# --- Corrupted run: replace subject tokens with noise ---
# "Eiffel Tower" spans tokens 1-2 ("The"=0, " Eiffel"=1, " Tower"=2)
torch.manual_seed(0)
corrupt_tokens = clean_tokens.clone()
subject_positions = [1, 2]   # positions of " Eiffel" and " Tower"
for pos in subject_positions:
    corrupt_tokens[0, pos] = torch.randint(1000, 10000, (1,)).item()

corrupt_logits, corrupt_cache = model.run_with_cache(corrupt_tokens)
corrupt_prob = torch.softmax(corrupt_logits[0, -1], dim=-1)[target_id].item()
print(f"Corrupt P('{TARGET_TOKEN}') = {corrupt_prob:.4f}")

In [ ]:
def patch_resid_and_measure(model, corrupt_tokens, clean_cache,
                            target_id, n_layers, seq_len):
    """
    For each (layer, position), patch the residual stream from clean_cache
    into a forward pass on corrupt_tokens and record the restored probability.
    Returns an (n_layers, seq_len) array.
    """
    results = np.zeros((n_layers, seq_len))

    for layer in range(n_layers):
        for pos in range(seq_len):

            # Closure captures current layer and pos
            def hook_fn(value, hook, _layer=layer, _pos=pos):
                value[0, _pos] = clean_cache['resid_post', _layer][0, _pos]
                return value

            hook_name = f'blocks.{layer}.hook_resid_post'
            patched_logits = model.run_with_hooks(
                corrupt_tokens,
                fwd_hooks=[(hook_name, hook_fn)]
            )
            prob = torch.softmax(patched_logits[0, -1], dim=-1)[target_id].item()
            # Normalise: 0 = no improvement, 1 = full recovery
            results[layer, pos] = (prob - corrupt_prob) / (clean_prob - corrupt_prob + 1e-8)

    return results

print("Running activation patching (this may take ~1-2 min on CPU)...")
patch_results = patch_resid_and_measure(
    model, corrupt_tokens, clean_cache,
    target_id, model.cfg.n_layers, seq_len
)
print("Done.")

In [ ]:
# --- Visualize causal tracing heatmap ---
fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(patch_results, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(seq_len))
ax.set_xticklabels(clean_token_strs, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(model.cfg.n_layers))
ax.set_yticklabels([f'L{l}' for l in range(model.cfg.n_layers)])
ax.set_xlabel('Token position')
ax.set_ylabel('Layer (residual stream patched)')
ax.set_title(f'Causal Tracing: Normalised probability recovery of "{TARGET_TOKEN}"\n'
             f'(green=full recovery, red=no recovery)')
plt.colorbar(im, ax=ax, label='Normalised recovery')

for l in range(model.cfg.n_layers):
    for p in range(seq_len):
        v = patch_results[l, p]
        if v > 0.3:
            ax.text(p, l, f'{v:.2f}', ha='center', va='center', fontsize=6,
                    color='black')

plt.tight_layout()
plt.show()

# Summarise the most important positions
best_pos = patch_results.max(axis=0)
print("\nMost causally important positions:")
for p in np.argsort(best_pos)[::-1][:5]:
    print(f"  Position {p} ({repr(clean_token_strs[p])})  max_recovery={best_pos[p]:.3f}")

---
## 5. Sparse Autoencoder (SAE) Features

A **Sparse Autoencoder** learns an over-complete dictionary of features that sparsely reconstruct MLP activations.

The idea: MLP hidden states are **superpositioned** — many concepts packed into fewer neurons via linear combinations. An SAE with `n_features >> d_mlp` disentangles these into one-concept-per-feature.

**Architecture**: `encode → ReLU (sparse) → decode`  
**Loss**: reconstruction MSE + L1 sparsity penalty on activations

We train a small SAE on layer 8's MLP output activations collected from a mini-corpus, then inspect the learned features.

In [ ]:
import torch.nn as nn

class SparseAutoencoder(nn.Module):
    """
    Simple tied-weight SAE.
    d_in      : activation dimension (d_mlp for GPT-2 = 3072)
    n_features: dictionary size (typically 4-8x d_in)
    l1_coeff  : sparsity penalty weight
    """
    def __init__(self, d_in: int, n_features: int, l1_coeff: float = 1e-3):
        super().__init__()
        self.d_in = d_in
        self.n_features = n_features
        self.l1_coeff = l1_coeff

        self.W_enc = nn.Parameter(torch.empty(d_in, n_features))
        self.b_enc = nn.Parameter(torch.zeros(n_features))
        self.W_dec = nn.Parameter(torch.empty(n_features, d_in))
        self.b_dec = nn.Parameter(torch.zeros(d_in))

        nn.init.kaiming_uniform_(self.W_enc)
        # Initialise decoder as transpose of encoder (tied-weight start)
        self.W_dec.data = self.W_enc.data.T.clone()
        self._normalise_dec()

    def _normalise_dec(self):
        """Keep decoder columns unit-norm (prevents trivial solutions)."""
        with torch.no_grad():
            norms = self.W_dec.norm(dim=1, keepdim=True).clamp(min=1e-6)
            self.W_dec.data = self.W_dec.data / norms

    def encode(self, x):
        return torch.relu(x @ self.W_enc + self.b_enc)

    def decode(self, z):
        return z @ self.W_dec + self.b_dec

    def forward(self, x):
        z = self.encode(x)          # [batch, n_features]  — sparse
        x_hat = self.decode(z)      # [batch, d_in]        — reconstruction
        mse  = (x - x_hat).pow(2).mean()
        l1   = self.l1_coeff * z.abs().mean()
        loss = mse + l1
        return loss, x_hat, z

print("SparseAutoencoder defined.")

In [ ]:
# --- Collect MLP output activations from a mini-corpus ---
SAE_LAYER = 8   # layer whose MLP outputs we will analyse

CORPUS = [
    "The Eiffel Tower is located in Paris, the capital of France.",
    "London is the capital of the United Kingdom and sits on the River Thames.",
    "Tokyo is a densely populated city in Japan known for technology and culture.",
    "Rome is the eternal city, famous for ancient history and the Vatican.",
    "New York City, often called the Big Apple, is a global financial hub.",
    "The Great Wall of China stretches thousands of miles across northern China.",
    "Shakespeare was an English playwright who wrote Hamlet and Macbeth.",
    "Einstein developed the theory of general relativity in the early 20th century.",
    "Python is a popular programming language known for readability and simplicity.",
    "The Amazon rainforest is the world's largest tropical rainforest in South America.",
    "The Pacific Ocean is the largest and deepest ocean on Earth.",
    "Beethoven composed nine symphonies and went deaf later in his life.",
    "DNA contains the genetic instructions for the development of all living organisms.",
    "The United Nations was founded in 1945 after the Second World War.",
    "Gravity is a fundamental force that attracts objects with mass toward each other.",
    "The Louvre in Paris houses the Mona Lisa painted by Leonardo da Vinci.",
]

mlp_acts_list = []
hook_name = f'blocks.{SAE_LAYER}.hook_mlp_out'

for text in CORPUS:
    toks = model.to_tokens(text)
    _, c = model.run_with_cache(toks, names_filter=hook_name)
    acts = c[hook_name][0]   # [seq_len, d_mlp]
    mlp_acts_list.append(acts.detach().cpu())

mlp_acts = torch.cat(mlp_acts_list, dim=0)   # [total_tokens, d_mlp]
print(f"Collected {mlp_acts.shape[0]} token activations  |  d_mlp = {mlp_acts.shape[1]}")

In [ ]:
# --- Train the SAE ---
D_IN       = mlp_acts.shape[1]     # 3072 for GPT-2 small
N_FEATURES = D_IN * 4              # 4x over-complete dictionary
L1_COEFF   = 2e-4
EPOCHS     = 200
LR         = 1e-3
BATCH_SIZE = 64

sae = SparseAutoencoder(D_IN, N_FEATURES, L1_COEFF).to(device)
optimiser = torch.optim.Adam(sae.parameters(), lr=LR)
dataset   = mlp_acts.to(device)

losses, mse_track, l1_track, sparsity_track = [], [], [], []

for epoch in range(EPOCHS):
    perm = torch.randperm(dataset.shape[0])
    epoch_loss = 0.0

    for i in range(0, dataset.shape[0], BATCH_SIZE):
        batch = dataset[perm[i:i+BATCH_SIZE]]
        optimiser.zero_grad()
        loss, x_hat, z = sae(batch)
        loss.backward()
        optimiser.step()
        sae._normalise_dec()
        epoch_loss += loss.item()

    # Track metrics every 20 epochs
    if epoch % 20 == 0 or epoch == EPOCHS - 1:
        with torch.no_grad():
            _, x_hat_all, z_all = sae(dataset)
            mse_val = (dataset - x_hat_all).pow(2).mean().item()
            l1_val  = z_all.abs().mean().item()
            sparsity = (z_all > 0).float().mean().item()  # fraction of active features
        print(f"Epoch {epoch:3d}  loss={epoch_loss:.4f}  MSE={mse_val:.4f}  "
              f"L1={l1_val:.4f}  sparsity={sparsity:.3f}")
        losses.append(epoch_loss)
        mse_track.append(mse_val)
        l1_track.append(l1_val)
        sparsity_track.append(sparsity)

print("Training complete.")

In [ ]:
# --- SAE training curves ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
epochs_x = [i * 20 for i in range(len(mse_track)-1)] + [EPOCHS - 1]

axes[0].plot(epochs_x, mse_track, marker='o', color='steelblue')
axes[0].set_title('Reconstruction MSE')
axes[0].set_xlabel('Epoch')

axes[1].plot(epochs_x, l1_track, marker='o', color='coral')
axes[1].set_title('Mean Feature Activation (L1)')
axes[1].set_xlabel('Epoch')

axes[2].plot(epochs_x, sparsity_track, marker='o', color='green')
axes[2].set_title('Feature Density (fraction active)')
axes[2].set_xlabel('Epoch')

for ax in axes:
    ax.grid(True, alpha=0.3)
plt.suptitle('SAE Training Curves', fontsize=13)
plt.tight_layout()
plt.show()

# --- Per-feature activation frequency ---
with torch.no_grad():
    z_all = sae.encode(dataset)
    freq = (z_all > 0).float().mean(dim=0).cpu().numpy()  # [n_features]

plt.figure(figsize=(12, 3))
plt.hist(freq, bins=100, color='purple', alpha=0.7)
plt.xlabel('Feature activation frequency')
plt.ylabel('Number of features')
plt.title('SAE Feature Activation Frequency Distribution\n(ideal: most features activate rarely — sparse)')
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Features never activated : {(freq == 0).sum()}")
print(f"Features active <1%      : {(freq < 0.01).sum()}")
print(f"Features active >50%     : {(freq > 0.5).sum()}")

In [ ]:
# --- Inspect top activating tokens for the 10 most frequent SAE features ---
# For each feature, find which tokens most strongly activate it and show context.

def top_activating_tokens(model, sae, texts, feature_idx, top_k=5, layer=SAE_LAYER):
    """
    For a given SAE feature index, find the (text, token, activation) triples
    with the highest feature activation.
    """
    all_entries = []
    hook_name = f'blocks.{layer}.hook_mlp_out'
    for text in texts:
        toks = model.to_tokens(text)
        tok_strs = model.to_str_tokens(text)
        _, c = model.run_with_cache(toks, names_filter=hook_name)
        acts = c[hook_name][0].detach()            # [seq_len, d_mlp]
        z = sae.encode(acts.to(device)).cpu()      # [seq_len, n_features]
        feat_acts = z[:, feature_idx]              # [seq_len]
        for pos, val in enumerate(feat_acts):
            all_entries.append((text, tok_strs[pos], pos, val.item()))

    all_entries.sort(key=lambda x: -x[3])
    return all_entries[:top_k]

# Pick the top-10 most frequently active features
top_features = np.argsort(freq)[::-1][:10]

print("Top activating tokens for the 10 most active SAE features:\n")
for feat_idx in top_features[:5]:
    top_tokens = top_activating_tokens(model, sae, CORPUS, feat_idx, top_k=5)
    print(f"Feature {feat_idx:5d}  (freq={freq[feat_idx]:.3f})")
    for text, tok, pos, val in top_tokens:
        snippet = text[:60] + ('...' if len(text) > 60 else '')
        print(f"   {repr(tok):15s}  act={val:.3f}  [{snippet}]")
    print()

In [ ]:
# --- PCA of SAE feature activations coloured by corpus sentence ---
with torch.no_grad():
    z_all_np = sae.encode(dataset).cpu().numpy()  # [total_tokens, n_features]

# Assign each token a sentence index
sentence_ids = []
for s_idx, text in enumerate(CORPUS):
    toks = model.to_tokens(text)
    sentence_ids.extend([s_idx] * toks.shape[1])
sentence_ids = np.array(sentence_ids[:z_all_np.shape[0]])

pca = PCA(n_components=2)
z_2d = pca.fit_transform(z_all_np)

plt.figure(figsize=(10, 7))
scatter = plt.scatter(z_2d[:, 0], z_2d[:, 1],
                      c=sentence_ids, cmap='tab20',
                      alpha=0.6, s=15)
plt.colorbar(scatter, label='Sentence index')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.title('PCA of SAE feature activations — tokens coloured by source sentence\n'
          '(clusters suggest the SAE has captured sentence-level structure)')
plt.tight_layout()
plt.show()

print("\nSummary")
print("="*60)
print(f"SAE dictionary size   : {N_FEATURES}")
print(f"Features that activate: {(freq > 0).sum()} / {N_FEATURES}")
print(f"Mean features active  : {(z_all_np > 0).sum(axis=1).mean():.1f} per token")